# 07 — Hallucination Detection

Detecting hallucinations automatically enables quality filtering in LLM pipelines.

## SelfCheckGPT, NLI-Based Checking, Retrieval Verification

**SelfCheckGPT** (Manakul et al., 2023): sample *k* responses, then for each sentence in the main response, check whether the other sampled responses are consistent with it. Inconsistency signals hallucination.

**NLI-based checking**: split the response into atomic claims, then verify each claim using a natural language inference (NLI) model:
1. Claim: 'The Eiffel Tower was built in 1887'
2. Source: Wikipedia passage about Eiffel Tower
3. NLI: Does the source **entail** the claim? → No → potential hallucination

**Retrieval verification**: for factual claims, retrieve relevant documents and check if they support the claim. Used in RAG pipelines to measure 'faithfulness' (does the response stay within the retrieved context?).

**FActScore** (Min et al., 2023): decomposes a long-form generation into atomic facts and computes the fraction supported by a reference knowledge source.

**FactCC / SummaC**: specific to summarisation — NLI-based consistency checking between summary and source document.

In [ ]:
# SelfCheckGPT pipeline + NLI verifier
import numpy as np

class SimulatedLLM:
    def __init__(self, hallucination_rate=0.3):
        self.hallucination_rate = hallucination_rate

    def generate(self, question, sample_id=0):
        """Generate a response. Sometimes hallucinates facts."""
        np.random.seed(hash(question) % 10000 + sample_id)
        sentences = []
        facts = {
            'paris': 'Paris is the capital of France.',
            'eiffel': 'The Eiffel Tower was completed in 1889.',
            'python': 'Python was created by Guido van Rossum in 1991.',
        }
        hallucinations = {
            'paris': 'Paris is the capital of Belgium.',
            'eiffel': 'The Eiffel Tower was completed in 1850.',
            'python': 'Python was created by James Gosling in 1985.',
        }

        for key in ['paris', 'eiffel', 'python']:
            if np.random.random() < self.hallucination_rate:
                sentences.append((hallucinations[key], False))
            else:
                sentences.append((facts[key], True))

        return sentences

llm = SimulatedLLM(hallucination_rate=0.3)

def self_check_gpt(question, k=5):
    """SelfCheckGPT: check consistency across k samples."""
    # Main sample
    main_response = llm.generate(question, sample_id=0)
    # k-1 additional samples
    other_samples = [llm.generate(question, sample_id=i+1) for i in range(k-1)]

    results = []
    for sentence, is_true in main_response:
        # Count how many other samples contain consistent information
        consistent_count = 0
        for sample in other_samples:
            # Check if any sentence in this sample is consistent (simplified: same sentence)
            sample_sents = [s for s, _ in sample]
            if sentence in sample_sents:
                consistent_count += 1

        consistency = consistent_count / (k - 1)
        predicted_hallucination = consistency < 0.5
        results.append({
            'sentence': sentence[:60],
            'is_true': is_true,
            'consistency': consistency,
            'predicted_hallucination': predicted_hallucination
        })

    return results

# Run SelfCheckGPT on a question
question = 'Tell me about Paris and the Eiffel Tower and Python.'
results = self_check_gpt(question, k=10)

print('SelfCheckGPT results:')
print(f'{"Sentence":<65} {"True":>6} {"Consist.":>9} {"Predicted":>10}')
for r in results:
    print(f'{r["sentence"]:<65} {str(r["is_true"]):>6} {r["consistency"]:>9.2f} {str(r["predicted_hallucination"]):>10}')

In [ ]:
# Evaluation: precision/recall of hallucination detection
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

all_results = []
for q_idx in range(50):
    q = f'question_{q_idx}'
    res = self_check_gpt(q, k=8)
    all_results.extend(res)

y_true = [1 - int(r['is_true']) for r in all_results]  # 1=hallucination
y_pred = [int(r['predicted_hallucination']) for r in all_results]

print('Hallucination detection performance:')
print(classification_report(y_true, y_pred, target_names=['correct', 'hallucination']))

# Consistency score distribution
fig, ax = plt.subplots(figsize=(8, 4))
true_consis = [r['consistency'] for r in all_results if r['is_true']]
false_consis = [r['consistency'] for r in all_results if not r['is_true']]
ax.hist(true_consis, bins=10, alpha=0.7, label='Correct', color='steelblue')
ax.hist(false_consis, bins=10, alpha=0.7, label='Hallucination', color='tomato')
ax.set_xlabel('SelfCheckGPT consistency score')
ax.set_ylabel('Count')
ax.set_title('SelfCheckGPT: Consistency Score Distribution')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/selfcheck_gpt.png', dpi=100, bbox_inches='tight')
plt.show()
print('SelfCheckGPT analysis saved')